In [17]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import make_column_selector,make_column_transformer
from sklearn.neighbors import KNeighborsClassifier
os.chdir("E:/Machine_Learning/Datasets")

In [18]:
loan = pd.read_csv('loan.csv',index_col=0)
loan.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
Loan_ID,,,,,,,,,,,,
LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [19]:
X , y = loan.drop('Loan_Status',axis=1),loan['Loan_Status']

In [40]:
imp_categorical = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')

imp_numerical = SimpleImputer().set_output(transform='pandas')

ct_imp = make_column_transformer((imp_categorical,make_column_selector(dtype_include='object')),
                                (imp_numerical,make_column_selector(dtype_include=['int64','float64'])),
                                verbose_feature_names_out=False)
ct_imp = ct_imp.set_output(transform='pandas')

In [43]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')

ct_ohe = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                                ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                                verbose_feature_names_out=False).set_output(transform='pandas')

In [44]:
scaler = StandardScaler()

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

ks = np.arange(1,16)
scores = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    pipe = Pipeline([('IMP',ct_imp),("OHE",ct_ohe),('SCL',scaler),('KNN',knn)])
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    scores.append([k,acc])
df_scores = pd.DataFrame(scores,columns=['k','Accuracy Score'])
df_scores.nlargest(1,'Accuracy Score')

,k,Accuracy Score
5,6,0.843243
